# 05 Subhalo training

Trains one subhalo config (real space, TNG300-3) and scores it on the test slab. One run per Colab session: pick the config in the second cell and run everything.

| Config | What it tests |
|---|---|
| `A_scratch_L10` | From scratch at spacing 10 h^-1 Mpc |
| `B_finetune_L10` | The dark-matter baseline fine-tuned at spacing 10 |
| `C_mixed` | One model on spacings 3, 5, 7 and 10 at once, not told which |
| `C_film` | As C, told the spacing (FiLM, feature-wise linear modulation) |
| `C_film_curriculum` | As C_film, with denser fields favoured early on |
| `A_smooth2ch_L10` | A with a second, smoothed input channel |
| `B_smooth2ch_L10` | B with the smoothed channel |
| `C_mixed_smooth2ch` | C_mixed with the smoothed channel |
| `C_mixed_smooth2ch_depth5` | C_mixed_smooth2ch with one more U-Net level (more reach) |
| `C_mixed_smooth2ch_voidloss` | C_mixed_smooth2ch with an added void-vs-rest loss term |
| `C_mixed_smooth2ch_deepsup` | C_mixed_smooth2ch with deep supervision |
| `C_mixed_smooth2ch_depth5_deepsup` | depth 5 with deep supervision |
| `A_smoothonly_L10` | Ablation: smoothed channel only, no raw channel |
| `A256_scratch_L10` | A on a 256³ grid (more context per crop), scored at 512³; needs notebook 07 first |
| `papersplit_L10_copy`, `papersplit_L10_region` | Split test at spacing 10: the first paper's subcube split, and the same with rotations kept together (`configs/research/papersplit/`) |
| `papersplit_DM_copy`, `papersplit_DM_region` | The same pair on the full dark-matter density |
| `C_mixed_smooth2ch_depth5_half` | Data-limit test: C_mixed_smooth2ch_depth5 trained on half the training slab, same validation and test slabs (`configs/research/datalimit/`) |
| `C_mixed_smooth2ch_depth5_rsd` | C_mixed_smooth2ch_depth5 on redshift-space inputs (z as line of sight, `rotate: z`), real-space labels |
| `C_mixed_smooth2ch_depth5_rsd_stripz` | The redshift-space run with strip pooling along z (the line of sight) after the bottleneck |
| `C_mixed_smooth2ch_depth5_rsd_stripx` | Control: the same block along x, across the line of sight |
| `C_mixed_smooth2ch_depth5_rsd_yz` | The redshift-space run on two lines of sight, z and y (turned onto z); needs the y grids from notebook 13 |
| `dm_smooth2ch_depth5` | Dark-matter model with the depth-5, two-channel shape of the subhalo model, for starting it from (and later distillation) |
| `C_mixed_smooth2ch_depth5_dmpre` | `C_mixed_smooth2ch_depth5` started from `dm_smooth2ch_depth5` (learning rate 2e-4) |
| `C_mixed_smooth2ch_depth5_lr2e-4` | Control: the same from random weights (it scored the same as `_dmpre`: the lower rate, not the start, cost ~0.01) |
| `C_mixed_smooth2ch_depth5_distill` | `C_mixed_smooth2ch_depth5` (1e-3, from scratch) also learning `dm_smooth2ch_depth5`'s class scores (distillation, `configs/research/distill/`) |
| `C_mixed_smooth2ch_unetr` | `C_mixed_smooth2ch_depth5` with a vision transformer encoder (UNETR, experimental; `configs/research/vit/`) |
| `C_mixed_smooth2ch_unetr_b4` | As `C_mixed_smooth2ch_unetr` with batch 4 (400 steps per epoch), if batch 8 runs out of memory |
| `C_mixed_smooth2ch_unetr_lr3e-4` | `C_mixed_smooth2ch_unetr` with learning rate 3e-4 |
| `C_mixed_smooth2ch_unetr_convstem_lr3e-4` | `C_mixed_smooth2ch_unetr_lr3e-4` with a convolutional stem (stride-2 convolutions in place of the patch convolution) |

To repeat a run with another seed (to measure run-to-run spread), set `SEED`, e.g. `CONFIG = 'C_mixed_smooth2ch'` with `SEED = 13`, then 14.

Suggested order: A and B (short), then C and C_film, then C_film_curriculum. Run `04_subhalo_smoke_test` once before the first of these.

Runtime: **L4 or A100 GPU** with **High-RAM**. If the session ends early, rerun from the top with the same config: the run resumes after its last completed epoch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/deepvoid
!git clone -q "https://github.com/skumagai3/deepvoid.git" /content/deepvoid
!pip -q install --force-reinstall --no-deps /content/deepvoid
!git -C /content/deepvoid log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
CONFIG = 'C_mixed_smooth2ch_depth5'   # A_scratch_L10, B_finetune_L10, C_mixed, C_film, C_film_curriculum, A_smooth2ch_L10,
           # B_smooth2ch_L10, C_mixed_smooth2ch, C_mixed_smooth2ch_depth5,
           # C_mixed_smooth2ch_voidloss, C_mixed_smooth2ch_deepsup,
           # C_mixed_smooth2ch_depth5_deepsup, A_smoothonly_L10, A256_scratch_L10,
           # papersplit_L10_copy, papersplit_L10_region, papersplit_DM_copy, papersplit_DM_region,
           # C_mixed_smooth2ch_depth5_rsd, C_mixed_smooth2ch_depth5_rsd_stripz,
           # C_mixed_smooth2ch_depth5_rsd_stripx, C_mixed_smooth2ch_depth5_rsd_yz,
           # dm_smooth2ch_depth5, C_mixed_smooth2ch_depth5_dmpre, C_mixed_smooth2ch_depth5_lr2e-4,
           # C_mixed_smooth2ch_depth5_distill, C_mixed_smooth2ch_unetr,
           # C_mixed_smooth2ch_unetr_b4, C_mixed_smooth2ch_unetr_lr3e-4,
           # C_mixed_smooth2ch_unetr_convstem_lr3e-4, C_mixed_smooth2ch_depth5_half

SEED = None   # None: the seed in the config. A number trains a copy in <name>_seed<number>/

folder = ('papersplit' if CONFIG.startswith('papersplit_')
          else 'rsd' if '_rsd' in CONFIG
          else 'pretrain' if CONFIG.startswith('dm_') or CONFIG.endswith(('_dmpre', '_lr2e-4'))
          else 'distill' if CONFIG.endswith('_distill')
          else 'vit' if '_unetr' in CONFIG
          else 'datalimit' if CONFIG.endswith('_half')
          else 'subhalo')
config_path = f'/content/deepvoid/configs/research/{folder}/{CONFIG}.yaml'
print(open(config_path).read())
if SEED is not None:
    print(f'The seed in the file is overridden: training uses seed {SEED}, saved as <name>_seed{SEED}.')

## 1. Watch progress

TensorBoard opens with three charts pinned at the top: training loss, validation MCC and learning rate (Keras tags `epoch_loss`, `epoch_mcc`, `epoch_learning_rate`). Pins live in the page address, so the link below keeps them too. For the mixed runs, TensorBoard and `log.csv` also have `val_mcc_L3` ... `val_mcc_L10`.


In [ ]:
import json

from IPython.display import Javascript, display
from tensorboard import program

LOGDIR = '/tmp/deepvoid_tensorboard'
PINNED = ['epoch_loss', 'epoch_mcc', 'epoch_learning_rate']    # loss, validation MCC, learning rate

if 'TB_PORT' not in globals():                                    # start once per runtime
    tb = program.TensorBoard()
    tb.configure(argv=[None, '--logdir', LOGDIR])
    TB_PORT = int(tb.launch().rstrip('/').rsplit(':', 1)[1])

# The same embedding %tensorboard uses in Colab, plus the pinned charts in the address.
pins = json.dumps([{'plugin': 'scalars', 'tag': t} for t in PINNED], separators=(',', ':'))
display(Javascript('''
(async () => {
    const url = new URL(await google.colab.kernel.proxyPort(%d, {'cache': true}));
    url.searchParams.set('tensorboardColab', 'true');
    url.searchParams.set('pinnedCards', %s);
    const iframe = document.createElement('iframe');
    iframe.src = url;
    iframe.setAttribute('width', '100%%');
    iframe.setAttribute('height', '900');
    iframe.setAttribute('frameborder', 0);
    document.body.appendChild(iframe);
})();
''' % (TB_PORT, json.dumps(pins))))


## 2. Train, then score the test slab

In [ ]:
from deepvoid.train import run
out = run(config_path, seed=SEED)

## 3. Results

In [ ]:
from deepvoid.config import input_label
import json
import pandas as pd
import matplotlib.pyplot as plt

scores = json.load(open(out / 'test_scores.json'))
if any('validation' in s for s in scores.values()):
    # Paper-split runs: validation subcubes against held-out test subcubes.
    keys = ['mcc', 'void_mcc', 'void_f1', 'wall_f1', 'filament_f1', 'halo_f1']
    table = pd.DataFrame({f'{name} {part}': {k: s[part][k] for k in keys}
                          for name, s in scores.items() for part in ('validation', 'test')}).T
    print(table.round(3))
    for name, s in scores.items():
        print(f"{name}: {s['validation_with_a_copy_in_training']:.0%} of validation subcubes "
              'have a rotated copy of their region in training')
else:
    table = pd.DataFrame({input_label(name): {k: v for k, v in s.items() if k != 'confusion_matrix'}
                          for name, s in scores.items()}).T
    print(table[['mcc', 'void_f1', 'wall_f1', 'filament_f1', 'halo_f1']].round(3))

log = pd.read_csv(out / 'log.csv')
per_spacing = [c for c in log.columns if c.startswith('val_mcc_')]
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].plot(log.epoch, log.loss, label='train'); ax[0].plot(log.epoch, log.val_loss, label='val'); ax[0].set_title('loss'); ax[0].legend()
ax[1].plot(log.epoch, log.val_mcc, 'k', label='all')
for c in per_spacing:
    ax[1].plot(log.epoch, log[c], label=c[len('val_mcc_'):])
ax[1].set_title('validation MCC'); ax[1].legend()
ax[2].plot(log.epoch, log.learning_rate); ax[2].set_title('learning rate')
fig.suptitle(out.name)
plt.tight_layout()
from deepvoid.report import Report
Report(out).show(fig, 'training_curves')   # also saved in the run folder

Scores (`test_scores.json`), the test figure (`test_figure.png`) and the training curves (`training_curves.png`) are saved in the run folder.

## Shut down

Deletes this runtime once every cell above has finished, so it stops using compute units. The results are already on Drive; anything kept only under `/content` goes with the runtime. With **Run all**, Colab stops at the first cell that fails, so after an error the runtime stays up and you can look at it. Set `SHUT_DOWN = False` to keep the runtime.

In [ ]:
SHUT_DOWN = True   # False keeps the runtime after the run
if SHUT_DOWN:
    import time
    from google.colab import drive, runtime
    drive.flush_and_unmount()   # finish writing to Drive first
    time.sleep(5)
    runtime.unassign()          # disconnects and deletes this runtime